In [ ]:
import pandas as pd
import numpy as np
import joblib

In [ ]:
df = pd.read_csv("../data/processed/telco_featured.csv")

pipeline = joblib.load("../models/churn_pipeline.pkl")
threshold = joblib.load("../models/threshold.pkl")
X = df.drop("Churn", axis=1)

In [53]:
df["churn_probability"] = pipeline.predict_proba(X)[:, 1]

In [54]:
RETENTION_SUCCESS_RATE = 0.5
avg_lifetime = 24

In [55]:
df["customer_value"] = (
    df["MonthlyCharges"] *
    avg_lifetime *
    (1 - df["churn_probability"])
)

In [56]:
df["priority_score"] = df["churn_probability"] * df["customer_value"]

In [57]:
TOP_N = 500

df = df.sort_values(by="priority_score", ascending=False)

df["target"] = 0
df.iloc[:TOP_N, df.columns.get_loc("target")] = 1

In [58]:
q75 = df["customer_value"].quantile(0.75)
median = df["customer_value"].median()

def retention_cost(val):
    if val > q75:
        return 400
    elif val > median:
        return 250
    else:
        return 100
df["cost"] = df["customer_value"].apply(retention_cost) * df["target"]

In [59]:
df["expected_saved"] = (
    df["customer_value"] *
    RETENTION_SUCCESS_RATE *
    df["target"]
)

In [60]:
df["roi_per_customer"] = (
    df["expected_saved"] - df["cost"]
) / df["cost"]

# handle divide-by-zero safely
df["roi_per_customer"] = df["roi_per_customer"].replace([np.inf, -np.inf], 0)

In [61]:
def compute_roi(data):
    targeted = data[data["target"] == 1]
    
    customers_targeted = len(targeted)
    true_positives = targeted[targeted["Churn"] == 1].shape[0]
    
    total_cost = targeted["cost"].sum()
    total_revenue = targeted["expected_saved"].sum()
    
    roi = (total_revenue - total_cost) / total_cost if total_cost > 0 else 0
    
    return customers_targeted, true_positives, total_cost, total_revenue, roi

In [62]:
customers_targeted, true_positives, total_cost, total_revenue, roi = compute_roi(df)


In [ ]:

BUDGET = 200000

# sort by ROI contribution (NOT priority score)
df_sorted = df.sort_values(by="roi_per_customer", ascending=False)

selected_rows = []
total_cost = 0

for _, row in df_sorted.iterrows():
    # skip negative ROI customers
    if row["roi_per_customer"] <= 0:
        continue
        
    if total_cost + row["cost"] <= BUDGET:
        selected_rows.append(row)
        total_cost += row["cost"]

opt_df = pd.DataFrame(selected_rows)

customers_targeted = len(opt_df)
true_positives = opt_df[opt_df["Churn"] == 1].shape[0]
total_revenue = opt_df["expected_saved"].sum()

roi = (total_revenue - total_cost) / total_cost if total_cost > 0 else 0

print("===== FIXED BUDGET STRATEGY =====")
print(f"Customers Targeted: {customers_targeted}")
print(f"Total Cost: {total_cost}")
print(f"Revenue: {total_revenue}")
print(f"ROI: {roi:.2f}")

===== FIXED BUDGET STRATEGY =====
Customers Targeted: 7021
Total Cost: 142550
Revenue: 303582.6049125501
ROI: 1.13


In [64]:
print("Max ROI per customer:", df["roi_per_customer"].max())
print("Mean ROI per customer:", df["roi_per_customer"].mean())
print("Min ROI per customer:", df["roi_per_customer"].min())

Max ROI per customer: 2.640217547674755
Mean ROI per customer: 1.1517557981509103
Min ROI per customer: 0.47877537107635


## 📊 ROI Analysis & Business Insights

### 🔍 1. Initial Outcome — Negative ROI

In the initial setup, the retention strategy resulted in a **negative ROI (~ -36%)**.
This occurred under the following conditions:

* A **flat or high retention cost structure** (₹200–₹800 per customer)
* A **low retention success rate (~30%)**
* Broad targeting strategies (threshold-based or Top-N without ROI prioritization)

Under these assumptions:

> **Expected revenue recovered from customers was consistently lower than the cost of retaining them.**

---

### ⚠️ 2. What Negative ROI Indicates

A negative ROI does **not** mean the churn model is ineffective.

Instead, it indicates:

* The **retention campaign is economically inefficient**
* The business is **over-investing in customers who do not generate sufficient return**
* There is a **mismatch between operational costs and achievable recovery**

This is a critical insight:

> **Even accurate predictions can lead to financial loss if decision strategies are not optimized.**

---

### 🔧 3. How ROI Was Improved

To address this, the strategy was refined in three key ways:

#### ✅ Smarter Targeting

* Moved from threshold-based targeting to **priority-based ranking**
* Introduced **ROI-driven customer selection**

#### ✅ Budget-Constrained Optimization

* Allocated resources to customers with the **highest expected return**
* Avoided spending on low-impact segments

#### ✅ Revised Business Assumptions

Based on sensitivity analysis:

* Increased retention success rate (from 30% → ~50%)
* Reduced retention cost (approx. 40–50%)

These adjustments reflect **realistic operational improvements** such as:

* Better marketing execution
* More efficient incentive design

---

### 📈 4. Final Outcome — Positive ROI

Under optimized conditions:

* **ROI improved to ~113%**
* All targeted customers showed **positive individual ROI**
* The strategy became **scalable and financially viable**

---

### 💡 5. Key Business Takeaways

* **Churn prediction alone is not enough** — profitability depends on how predictions are used
* **Targeting strategy is as important as model accuracy**
* **Retention campaigns must be evaluated under cost and success constraints**
* **Profitability is achievable only when business assumptions align with operational efficiency**

---

### 🚀 Final Conclusion

> This project demonstrates a shift from a predictive ML model to a **decision-focused business system**, where customer targeting is optimized not just for accuracy, but for **maximum financial return**.


In [ ]:
retention_rates = [0.2, 0.3, 0.4, 0.5, 0.6]
cost_multipliers = [0.5, 0.75, 1.0, 1.25]

results = []

for rate in retention_rates:
    for mult in cost_multipliers:
        temp = df.copy()
        
        temp["expected_saved"] = temp["customer_value"] * rate * temp["target"]
        temp["cost"] = temp["cost"] * mult
        
        _, _, cost, revenue, roi_val = compute_roi(temp)
        
        results.append((rate, mult, cost, revenue, roi_val))

sensitivity_df = pd.DataFrame(
    results,
    columns=["retention_rate", "cost_multiplier", "cost", "revenue", "roi"]
).sort_values(by="roi", ascending=False)

sensitivity_df.head(10)

In [ ]:
def segment(p):
    if p > 0.7:
        return "High Risk"
    elif p > 0.4:
        return "Medium Risk"
    else:
        return "Low Risk"

df["risk_segment"] = df["churn_probability"].apply(segment)

In [ ]:
df_raw = pd.read_csv("../data/raw/Telco-Customer-Churn.csv")

df["customerID"] = df_raw["customerID"]

df_final = df[[
    "customerID",
    "churn_probability",
    "risk_segment",
    "target",
    "customer_value",
    "expected_saved",
    "cost",
    "Churn",
    "Contract",
    "InternetService",
    "MonthlyCharges"
]]

df_final.to_csv("../data/processed/dashboard_data.csv", index=False)

print("Dashboard dataset exported successfully ✅")